# OASIS Infobyte — Data Science Task 3: Car Price Prediction

**Goal:** predict the selling price of a used car from information such as brand, car age, kilometres driven, fuel type and transmission.

This notebook follows the task checklist: data cleaning, feature engineering, exploratory data analysis, categorical encoding, an 80/20 train-test split, two regression models, MAE/RMSE/R² evaluation, and a feature-importance chart.

## Before you run the notebook

1. Download a CarDekho vehicle dataset. The commonly used Kaggle dataset is [Vehicle Dataset from CarDekho](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho). You may also have a file named `CAR DETAILS FROM CAR DEKHO.csv`.
2. Put the CSV file in the **same folder** as this notebook, or change `DATA_PATH` in the next cell to its full path.
3. Install the libraries in `requirements.txt` if they are not installed.
4. Run the cells from top to bottom.

**Important about price units:** different CarDekho files store prices differently. Some files use rupees; the small `car data.csv` dataset commonly stores prices in lakhs. This notebook preserves the units in your file, so interpret predictions using the same unit as `selling_price` in the CSV.

**Reproducibility note:** this notebook supports common column-name variants, including `Car_Name`/`name`, `Selling_Price`/`selling_price`, and `Kms_Driven`/`km_driven`. The optional `Present_Price` column is used only when it exists in your dataset.

## 1. Import the libraries

In [ ]:
# If needed, install dependencies in a notebook cell by running:
# %pip install pandas numpy matplotlib seaborn scikit-learn joblib

from pathlib import Path
from datetime import datetime
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load the dataset

In [ ]:
# Change this if your CSV has a different name or is in another folder.
DATA_PATH = "car data.csv"

# Try common filenames if the default file is not present.
possible_files = [
    DATA_PATH,
    "CAR DETAILS FROM CAR DEKHO.csv",
    "CAR DETAILS FROM CAR DEKHO (1).csv",
    "car_details.csv",
    "cardekho.csv",
]
existing_path = next((Path(p) for p in possible_files if Path(p).exists()), None)

if existing_path is None:
    raise FileNotFoundError(
        "CSV dataset not found. Download the CarDekho dataset, put it beside this notebook, "
        "and set DATA_PATH to its exact filename. Checked: " + ", ".join(possible_files)
    )

raw_df = pd.read_csv(existing_path)
print(f"Loaded: {existing_path}")
print(f"Rows: {raw_df.shape[0]:,} | Columns: {raw_df.shape[1]}")
display(raw_df.head())

## 3. Inspect and standardise column names

In [ ]:
print("Original columns:")
print(raw_df.columns.tolist())

# Make column names consistent, e.g. 'Selling_Price' -> 'selling_price'.
df = raw_df.copy()
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Map common dataset variants to consistent names.
aliases = {
    "car_name": ["car_name", "name", "car", "model"],
    "year": ["year", "manufacture_year", "manufacturing_year"],
    "selling_price": ["selling_price", "sellingprice", "price"],
    "present_price": ["present_price", "presentprice", "current_price", "ex_showroom_price"],
    "km_driven": ["km_driven", "kms_driven", "kilometers_driven", "kilometres_driven", "driven_km"],
    "fuel_type": ["fuel_type", "fuel", "fueltype"],
    "seller_type": ["seller_type", "sellertype", "seller"],
    "transmission": ["transmission", "transmission_type", "gearbox"],
    "owner": ["owner", "owners", "previous_owners", "no_of_owners"],
}
rename_map = {}
for standard_name, variants in aliases.items():
    for variant in variants:
        if variant in df.columns:
            rename_map[variant] = standard_name
            break

df = df.rename(columns=rename_map)
print("Standardised columns:")
print(df.columns.tolist())

required_columns = ["car_name", "year", "selling_price", "km_driven", "fuel_type", "transmission"]
missing_required = [c for c in required_columns if c not in df.columns]
if missing_required:
    raise ValueError(
        f"Dataset is missing required columns: {missing_required}. "
        f"Available columns are: {df.columns.tolist()}"
    )

print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values before cleaning:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print(f"\nExact duplicate rows: {df.duplicated().sum()}")

## 4. Clean the data and engineer features

In [ ]:
# Convert columns that should contain numbers. Invalid strings become NaN and are handled below.
numeric_candidates = ["year", "selling_price", "present_price", "km_driven"]
for column in numeric_candidates:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

# Owner is numeric in some versions (0, 1, 2...) and a text label in others
# ('First Owner', 'Second Owner'...). Preserve whichever representation the file uses.
if "owner" in df.columns:
    owner_as_number = pd.to_numeric(df["owner"], errors="coerce")
    non_null_owner_count = df["owner"].notna().sum()
    numeric_owner_count = owner_as_number.notna().sum()
    if non_null_owner_count > 0 and numeric_owner_count >= 0.90 * non_null_owner_count:
        df["owner"] = owner_as_number
    else:
        df["owner"] = df["owner"].astype("string").str.strip().str.lower()

# Remove exact duplicates first.
df = df.drop_duplicates().copy()

# A row without the target price or manufacturing year cannot train this model.
df = df.dropna(subset=["selling_price", "year"]).copy()

# Remove impossible/non-useful values. Keep plausible high prices and mileage rather than
# deleting outliers without evidence.
current_year = datetime.now().year
df = df[(df["selling_price"] > 0) & (df["year"] >= 1950) & (df["year"] <= current_year)].copy()
if "km_driven" in df.columns:
    df = df[df["km_driven"].isna() | (df["km_driven"] >= 0)].copy()
if "present_price" in df.columns:
    df = df[df["present_price"].isna() | (df["present_price"] > 0)].copy()

# Standardise text fields so 'Petrol', 'petrol ', etc. become the same category.
for column in ["car_name", "fuel_type", "transmission", "seller_type"]:
    if column in df.columns:
        df[column] = df[column].astype("string").str.strip()
        df[column] = df[column].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        df[column] = df[column].str.lower()

# Fill incomplete text values consistently instead of losing otherwise useful rows.
for column in ["car_name", "fuel_type", "transmission", "seller_type"]:
    if column in df.columns:
        df[column] = df[column].fillna("unknown")

# Extract the manufacturer/brand from the start of a car name. Multi-word brands are checked first.
known_brands = [
    "land rover", "force motors", "mahindra renault", "maruti suzuki", "mercedes-benz",
    "aston martin", "ashok leyland", "rolls royce", "volkswagen", "chevrolet", "mitsubishi",
    "mahindra", "maruti", "hyundai", "honda", "toyota", "renault", "datsun", "ford", "tata",
    "nissan", "skoda", "fiat", "volvo", "jaguar", "jeep", "kia", "mg", "bmw", "audi",
    "lexus", "porsche", "mini", "isuzu", "opel", "daewoo", "bentley", "maserati",
    "lamborghini", "ambassador"
]
known_brands = sorted(known_brands, key=len, reverse=True)

def extract_brand(car_name):
    """Extract a best-effort make/brand from the beginning of a car name."""
    name = str(car_name).strip().lower()
    for brand in known_brands:
        if name == brand or name.startswith(brand + " "):
            return brand.title() if brand not in {"bmw", "mg"} else brand.upper()
    return name.split()[0].title() if name else "Unknown"

df["brand"] = df["car_name"].apply(extract_brand)
df["car_age"] = current_year - df["year"]

# The target must not be imputed. Fill other numeric features using their median.
for column in ["km_driven", "present_price", "owner"]:
    if column in df.columns and pd.api.types.is_numeric_dtype(df[column]) and df[column].isna().any():
        if df[column].notna().any():
            df[column] = df[column].fillna(df[column].median())

# Treat owner as a category if its values are labels such as 'First Owner'; retain numeric owners as numeric.
if "owner" in df.columns and not pd.api.types.is_numeric_dtype(df["owner"]):
    df["owner"] = df["owner"].astype("string").str.strip().str.lower().fillna("unknown")

print(f"Rows after cleaning: {len(df):,}")
print(f"Rows removed: {len(raw_df) - len(df):,}")
print(f"Feature engineering reference year: {current_year}")
print("Optional present_price column available:", "present_price" in df.columns)
display(df[["car_name", "brand", "year", "car_age", "km_driven", "fuel_type", "transmission", "selling_price"]].head())
print("\nMissing values after cleaning:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

### What this cleaning step does

- Removes exact duplicate listings and rows missing the target selling price or manufacturing year.
- Converts key columns to numeric types and rejects impossible values such as negative selling prices or future manufacturing years.
- Standardises text categories so small spelling/case differences are less likely to become separate categories.
- Creates **`car_age`** from the manufacturing year and **`brand`** from the car name.
- Keeps genuine high-price/high-mileage observations rather than deleting outliers automatically.

Brand extraction is a best-effort rule based on the beginning of the car name. Review `df[['car_name', 'brand']]` if the dataset contains unusual or multi-word manufacturer names.

## 5. Exploratory Data Analysis (EDA)

In [ ]:
print("Dataset summary:")
display(df.describe(include="all").T)

plt.figure(figsize=(9, 5))
sns.histplot(data=df, x="selling_price", bins=30, kde=True, color="steelblue")
plt.title("Distribution of Used-Car Selling Prices")
plt.xlabel("Selling price (units used in the CSV)")
plt.ylabel("Number of listings")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="fuel_type", y="selling_price", color="lightsteelblue")
plt.title("Selling Price by Fuel Type")
plt.xlabel("Fuel type")
plt.ylabel("Selling price (units used in the CSV)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="car_age", y="selling_price", hue="transmission", alpha=0.75)
plt.title("Selling Price vs. Car Age")
plt.xlabel(f"Car age in years (calculated relative to {current_year})")
plt.ylabel("Selling price (units used in the CSV)")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numeric columns. Categorical columns are handled later by one-hot encoding.
numeric_for_corr = df.select_dtypes(include=np.number).columns.tolist()
if len(numeric_for_corr) >= 2:
    plt.figure(figsize=(10, 6))
    correlation = df[numeric_for_corr].corr(numeric_only=True)
    sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f", center=0, square=False)
    plt.title("Correlation Heatmap — Numeric Features")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns to create a correlation heatmap.")

### How to interpret these charts

Use the actual plots from your run to write the findings in your own words. Check whether selling prices are right-skewed, whether median prices differ by fuel type, and whether older cars tend to sell for less. Correlation shows linear relationships between numeric variables; it does **not** prove causation. If your file stores selling price in rupees, keep that in mind when describing chart values.

## 6. Select the features and target

In [ ]:
# Target: the price we want the model to predict.
target = "selling_price"

# Use the features requested in the task. Add present_price only if the chosen CSV contains it.
feature_candidates = ["brand", "car_age", "km_driven", "fuel_type", "transmission"]
if "seller_type" in df.columns:
    feature_candidates.append("seller_type")
if "owner" in df.columns:
    feature_candidates.append("owner")
if "present_price" in df.columns:
    feature_candidates.append("present_price")

feature_columns = [c for c in feature_candidates if c in df.columns]
model_df = df[feature_columns + [target]].copy()
model_df = model_df.dropna(subset=[target])
X = model_df[feature_columns]
y = model_df[target]

print("Target:", target)
print("Features:", feature_columns)
print("Number of rows used for modelling:", len(model_df))
print("\nFeature data types:")
display(X.dtypes.to_frame("dtype"))

## 7. Split the data and prepare the preprocessing pipeline

In [ ]:
# Keep a separate test set to estimate how well the models perform on unseen listings.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

# Determine numeric vs categorical fields from the training data.
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features),
], remainder="drop")

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows:  {len(X_test):,}")
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

### Why encode categories?

A regression algorithm expects numbers, but brand, fuel type and transmission are text categories. **One-Hot Encoding** turns each category into a separate 0/1 indicator. The preprocessing is placed inside a scikit-learn `Pipeline`, so it learns from the training set and can consistently process test data and future predictions.

## 8. Train and compare two regression models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
}

fitted_models = {}
results = []

for model_name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = float(np.sqrt(mean_squared_error(y_test, predictions)))
    r2 = r2_score(y_test, predictions)

    fitted_models[model_name] = pipeline
    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2,
    })

metrics_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(metrics_df.style.format({"MAE": "{:,.3f}", "RMSE": "{:,.3f}", "R²": "{:.3f}"}))

best_model_name = metrics_df.loc[0, "Model"]
best_model = fitted_models[best_model_name]
best_predictions = best_model.predict(X_test)
print(f"Best model by lowest test RMSE: {best_model_name}")

### Understanding the evaluation metrics

- **MAE (Mean Absolute Error):** the average absolute difference between predicted and actual selling prices. Lower is better.
- **RMSE (Root Mean Squared Error):** penalises large prediction errors more heavily than MAE. Lower is better.
- **R² score:** indicates how much variation in the test-set prices is explained by the model. Closer to 1 is generally better; a negative score means the model performs worse than predicting the test-set mean.

For this notebook, the model with the **lowest test RMSE** is selected. Do not assume Random Forest must win; the test results decide.

## 9. Visualise actual vs. predicted prices

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.75)
min_value = min(float(y_test.min()), float(np.min(best_predictions)))
max_value = max(float(y_test.max()), float(np.max(best_predictions)))
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.title(f"Actual vs. Predicted Selling Price — {best_model_name}")
plt.xlabel("Actual selling price")
plt.ylabel("Predicted selling price")
plt.tight_layout()
plt.show()

comparison = pd.DataFrame({
    "Actual selling price": y_test.to_numpy(),
    "Predicted selling price": best_predictions,
})
comparison["Absolute error"] = np.abs(comparison["Actual selling price"] - comparison["Predicted selling price"])
display(comparison.head(10).style.format("{:,.2f}"))

## 10. Feature importance for the selected model

In [ ]:
# Permutation importance works with both Linear Regression and Random Forest pipelines.
# It measures how much the model score worsens when one input feature is shuffled.
importance_result = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance (RMSE increase)": importance_result.importances_mean,
    "Std": importance_result.importances_std,
}).sort_values("Importance (RMSE increase)", ascending=False)

display(importance_df.style.format({"Importance (RMSE increase)": "{:.4f}", "Std": "{:.4f}"}))

plot_df = importance_df.sort_values("Importance (RMSE increase)", ascending=True)
plt.figure(figsize=(9, max(4, 0.45 * len(plot_df))))
plt.barh(plot_df["Feature"], plot_df["Importance (RMSE increase)"], xerr=plot_df["Std"])
plt.title(f"Permutation Feature Importance — {best_model_name}")
plt.xlabel("Increase in RMSE when feature is shuffled (larger means more useful)")
plt.ylabel("Input feature")
plt.tight_layout()
plt.show()

## 11. Try the model on an example car

This final cell demonstrates a prediction using one row from the test set. It is deliberately a real row from the held-out test set, so the notebook does not invent input values or category names. You can replace `example_car` with a new DataFrame containing the same input columns to try your own car information.

In [ ]:
example_car = X_test.iloc[[0]].copy()
example_actual = float(y_test.iloc[0])
example_predicted = float(best_model.predict(example_car)[0])

print("Example input:")
display(example_car)
print(f"Actual selling price:    {example_actual:,.2f} (CSV price units)")
print(f"Predicted selling price: {example_predicted:,.2f} (CSV price units)")
print(f"Absolute error:          {abs(example_actual - example_predicted):,.2f} (CSV price units)")

## 12. Save the model (optional)

In [ ]:
# Optional: save the full preprocessing + model pipeline so it can be loaded later.
# The model file is created in the current notebook folder.
import joblib
model_path = Path("best_car_price_model.joblib")
joblib.dump(best_model, model_path)
print(f"Saved best pipeline to: {model_path.resolve()}")

## 13. Write your findings before submitting

Add a Markdown cell at the end of your notebook and write findings based on your actual output. Do not copy generic conclusions or claim that a model is accurate unless the metrics support it.

Use this checklist:

1. **Dataset:** report its number of rows and columns before/after cleaning, the key fields, and the price units.
2. **Cleaning:** state how many duplicates/invalid rows were removed and how missing values were handled.
3. **EDA:** describe what the price distribution, fuel-type box plot, car-age scatter plot and correlation heatmap show.
4. **Models:** report Linear Regression and Random Forest MAE, RMSE and R² from your results table.
5. **Best model:** state which model had the lowest test RMSE and quote its metrics.
6. **Feature importance:** explain which features most affected the best model's errors.
7. **Limitations:** the result depends on the dataset size, age and coverage; prices in historical listings may not represent today's market.

## Final checklist

- [ ] Notebook runs from the first cell to the last without errors.
- [ ] Dataset source is acknowledged in the README.
- [ ] Add written observations between charts / in Markdown cells.
- [ ] Include the notebook, `README.md`, `requirements.txt`, and selected screenshots in your GitHub task folder.
- [ ] Record and post your demo video as required in the internship task list.